# 03 - SoilGrids Covariates Analysis

This notebook explores SoilGrids 2.0 global soil predictions as covariates.

## Objectives
- Understand SoilGrids layer availability and quality
- Compare with local SSURGO data
- Evaluate resolution and uncertainty
- Develop integration strategy for foundation model

## Data Source
- SoilGrids 2.0: https://soilgrids.org/
- 250m spatial resolution globally
- Machine learning predictions based on environmental covariates

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

try:
    import rasterio
    from rasterio.plot import show
except ImportError:
    print("rasterio not installed. Run: pip install rasterio")
    rasterio = None

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('../../data/raw/soilgrids')
RESULTS_DIR = Path('../../results/data_analysis')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. SoilGrids Layer Overview

In [ ]:
# SoilGrids layer metadata
SOILGRIDS_LAYERS = {
    'soc': {
        'name': 'Soil Organic Carbon',
        'unit': 'dg/kg (divide by 10 for g/kg)',
        'relevance': 'Primary target property, strong correlation with nutrients'
    },
    'phh2o': {
        'name': 'pH in H2O',
        'unit': 'pH×10 (divide by 10 for pH)',
        'relevance': 'Controls nutrient availability, key soil property'
    },
    'clay': {
        'name': 'Clay Content',
        'unit': 'g/kg (divide by 10 for %)',
        'relevance': 'CEC proxy, water holding, nutrient retention'
    },
    'sand': {
        'name': 'Sand Content',
        'unit': 'g/kg (divide by 10 for %)',
        'relevance': 'Drainage indicator, leaching potential'
    },
    'silt': {
        'name': 'Silt Content',
        'unit': 'g/kg (divide by 10 for %)',
        'relevance': 'Texture component, completes clay+sand+silt=100%'
    },
    'bdod': {
        'name': 'Bulk Density',
        'unit': 'cg/cm³ (divide by 100 for g/cm³)',
        'relevance': 'Compaction indicator, root growth, water movement'
    },
    'cfvo': {
        'name': 'Coarse Fragments',
        'unit': 'cm³/dm³ (divide by 10 for %)',
        'relevance': 'Stone content, reduces soil volume'
    },
    'nitrogen': {
        'name': 'Total Nitrogen',
        'unit': 'cg/kg (divide by 100 for g/kg)',
        'relevance': 'Nutrient pool, related to SOC'
    }
}

print("SoilGrids 2.0 Layers:")
print("=" * 70)
for layer, info in SOILGRIDS_LAYERS.items():
    print(f"\n{layer.upper()}: {info['name']}")
    print(f"  Unit: {info['unit']}")
    print(f"  Relevance: {info['relevance']}")

In [ ]:
# Depth intervals
DEPTH_INTERVALS = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm', '100-200cm']

print("\nDepth Intervals:")
for depth in DEPTH_INTERVALS:
    print(f"  - {depth}")

print("\nNote: Each layer available at each depth with mean and uncertainty")

## 2. Load Sample Data

In [ ]:
# Check for available files
if DATA_DIR.exists():
    tif_files = list(DATA_DIR.glob('*.tif'))
    print(f"Found {len(tif_files)} GeoTIFF files:")
    for f in tif_files[:10]:
        print(f"  - {f.name}")
    if len(tif_files) > 10:
        print(f"  ... and {len(tif_files) - 10} more")
else:
    print(f"Data directory not found: {DATA_DIR}")
    print("Run data/soilgrids_downloader.py to download data")
    tif_files = []

In [ ]:
# Load a sample layer
if tif_files and rasterio is not None:
    sample_file = tif_files[0]
    
    with rasterio.open(sample_file) as src:
        print(f"Sample file: {sample_file.name}")
        print(f"\nRaster properties:")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
        print(f"  Resolution: {src.res}")
        print(f"  Shape: {src.shape}")
        print(f"  Data type: {src.dtypes[0]}")
        print(f"  NoData value: {src.nodata}")

## 3. Visualize Layers

In [ ]:
# Visualize available layers
if tif_files and rasterio is not None:
    # Find unique layers
    layer_files = {}
    for f in tif_files:
        parts = f.stem.split('_')
        if len(parts) >= 3:
            layer = parts[0]
            if layer not in layer_files:
                layer_files[layer] = f
    
    n_layers = min(6, len(layer_files))
    if n_layers > 0:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for i, (layer, filepath) in enumerate(list(layer_files.items())[:n_layers]):
            with rasterio.open(filepath) as src:
                data = src.read(1)
                data = np.ma.masked_equal(data, src.nodata)
                
                im = axes[i].imshow(data, cmap='viridis')
                axes[i].set_title(f"{layer}: {SOILGRIDS_LAYERS.get(layer, {}).get('name', layer)}")
                axes[i].axis('off')
                plt.colorbar(im, ax=axes[i], shrink=0.7)
        
        # Hide unused axes
        for i in range(n_layers, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / 'soilgrids_layers.png', dpi=150)
        plt.show()

## 4. Uncertainty Analysis

In [ ]:
# SoilGrids provides uncertainty estimates
print("SoilGrids Uncertainty:")
print("=" * 50)
print("")
print("Each layer has '_uncertainty' variant providing:")
print("  - Prediction interval (5th to 95th percentile range)")
print("  - Can be used as confidence weighting in training")
print("")
print("Integration strategies:")
print("  1. Use mean predictions as features")
print("  2. Weight samples by inverse uncertainty")
print("  3. Include uncertainty as additional feature")
print("  4. Use for data augmentation during training")

In [ ]:
# Compare mean vs uncertainty
if tif_files and rasterio is not None:
    # Find matching mean/uncertainty pairs
    mean_files = [f for f in tif_files if 'mean' in f.stem]
    unc_files = [f for f in tif_files if 'uncertainty' in f.stem]
    
    if mean_files and unc_files:
        # Plot comparison
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        with rasterio.open(mean_files[0]) as src:
            mean_data = np.ma.masked_equal(src.read(1), src.nodata)
            im1 = axes[0].imshow(mean_data, cmap='viridis')
            axes[0].set_title(f"Mean: {mean_files[0].stem}")
            plt.colorbar(im1, ax=axes[0])
        
        # Find corresponding uncertainty file
        layer = mean_files[0].stem.replace('_mean', '')
        unc_match = [f for f in unc_files if layer in f.stem]
        
        if unc_match:
            with rasterio.open(unc_match[0]) as src:
                unc_data = np.ma.masked_equal(src.read(1), src.nodata)
                im2 = axes[1].imshow(unc_data, cmap='Reds')
                axes[1].set_title(f"Uncertainty: {unc_match[0].stem}")
                plt.colorbar(im2, ax=axes[1])
        
        plt.tight_layout()
        plt.show()

## 5. Comparison with SSURGO

In [ ]:
# SoilGrids vs SSURGO comparison
print("SoilGrids vs SSURGO Comparison:")
print("=" * 60)
print("")
print("| Attribute        | SoilGrids          | SSURGO              |")
print("|------------------|--------------------|---------------------|")
print("| Coverage         | Global             | USA only            |")
print("| Resolution       | 250m               | Variable (polygon)  |")
print("| Data type        | ML predictions     | Survey measurements |")
print("| Uncertainty      | Provided           | Not provided        |")
print("| Depth intervals  | 6 standard depths  | Horizon-based       |")
print("| Update frequency | Periodic           | Continuous          |")
print("")
print("Recommendations:")
print("  - Use SSURGO for US locations where available")
print("  - Use SoilGrids as global fallback")
print("  - SoilGrids useful as pre-training covariate globally")
print("  - Consider ensemble: SSURGO + SoilGrids for US")

## 6. Feature Extraction Strategy

In [ ]:
# Point extraction example
def extract_soilgrids_at_point(lon, lat, layer_dir):
    """Extract SoilGrids values at a point location."""
    layer_dir = Path(layer_dir)
    features = {}
    
    for layer in SOILGRIDS_LAYERS.keys():
        # Find mean file for 0-5cm depth
        pattern = f"{layer}_0-5cm_mean.tif"
        matches = list(layer_dir.glob(pattern))
        
        if matches and rasterio is not None:
            with rasterio.open(matches[0]) as src:
                try:
                    row, col = src.index(lon, lat)
                    window = rasterio.windows.Window(col, row, 1, 1)
                    value = src.read(1, window=window)[0, 0]
                    if value != src.nodata:
                        features[f'sg_{layer}'] = float(value)
                except:
                    pass
    
    return features

# Example extraction
if DATA_DIR.exists():
    sample_features = extract_soilgrids_at_point(-95.0, 40.0, DATA_DIR)
    print("Extracted SoilGrids features at (-95, 40):")
    for key, value in sample_features.items():
        print(f"  {key}: {value}")

## 7. Summary and Recommendations

In [ ]:
# Generate summary
summary = {
    'data_source': 'SoilGrids 2.0',
    'spatial_resolution': '250m',
    'coverage': 'Global',
    'layers': list(SOILGRIDS_LAYERS.keys()),
    'depth_intervals': DEPTH_INTERVALS,
    'recommendations': {
        'primary_layers': ['soc', 'phh2o', 'clay', 'sand', 'bdod'],
        'primary_depth': '0-5cm (topsoil)',
        'uncertainty_usage': 'Sample weighting and data augmentation',
        'integration': 'Use as spatial context features for pre-training'
    },
    'unit_conversions': {
        'soc': 'Divide by 10 for g/kg',
        'phh2o': 'Divide by 10 for pH',
        'clay/sand/silt': 'Divide by 10 for %',
        'bdod': 'Divide by 100 for g/cm³'
    }
}

print("=" * 50)
print("SOILGRIDS ANALYSIS SUMMARY")
print("=" * 50)
print(f"Resolution: {summary['spatial_resolution']}")
print(f"Coverage: {summary['coverage']}")
print(f"Layers: {len(summary['layers'])}")
print("\nRecommended Layers:")
for layer in summary['recommendations']['primary_layers']:
    print(f"  - {layer}: {SOILGRIDS_LAYERS[layer]['name']}")
print("\nUnit Conversions:")
for var, conv in summary['unit_conversions'].items():
    print(f"  - {var}: {conv}")

In [ ]:
# Save summary
import json

with open(RESULTS_DIR / 'soilgrids_features.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to {RESULTS_DIR / 'soilgrids_features.json'}")